# Notebook ١٢: Retrieval و RAG

بوت يجاوب من أخبار لاعبي FPL الحقيقية: بحث TF-IDF، بحث بالمعنى (embeddings)، تركيب الـ prompt، وتوليد. كل خلية اتشغلت فعلًا.

## ١. قاعدة المعرفة

كل RAG بيبدأ بمستندات. هنا: أخبار الإصابات والانتقالات لكل لاعب في FPL.

In [ ]:
# قراءة JSON
import json
# بيانات FPL
d = json.load(open("fpl.json"))
# مستند لكل لاعب عنده خبر
docs = [p["web_name"] + ": " + p["news"] for p in d["elements"] if p["news"]]
# عدد المستندات
print(len(docs))
# أول 3 أمثلة
print(docs[:3])


## ٢. بحث بالكلمات (TF-IDF)

الأسرع والأبسط. بيدور على كلمات مشتركة بس.

In [ ]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
# المصفوفات
import numpy as np
# نحول النص لأرقام
tf = TfidfVectorizer(stop_words="english")
# مصفوفة المستندات
X = tf.fit_transform(docs)
# دالة البحث
def search_tfidf(q, k=3):
# تشابه السؤال بكل مستند
    s = (X @ tf.transform([q]).T).toarray().ravel()
# أعلى k مستندات
    return [docs[i] for i in s.argsort()[::-1][:k]]
# سؤالين
for q in ["who has a hamstring injury", "which player moved to another club"]:
# النتايج
    print(q, "->", search_tfidf(q))


## ٣. بحث بالمعنى (Dense)

embeddings بتفهم المعنى حتى لو الكلمات مختلفة.

In [ ]:
# PyTorch
import torch
# تحميل الموديل
from transformers import AutoTokenizer, AutoModel
# tokenizer
tk = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
# موديل الـ embeddings
em = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2").eval()
# متجه لكل نص
def embed(texts):
# النتايج
    out = []
# دفعات من 32
    for i in range(0, len(texts), 32):
# نشفّر
        enc = tk(texts[i:i+32], padding=True, truncation=True, return_tensors="pt")
# من غير gradients
        with torch.no_grad():
# متجهات التوكنز
            h = em(**enc).last_hidden_state
# نتجاهل الحشو
        m = enc["attention_mask"].unsqueeze(-1)
# المتوسط
        v = (h * m).sum(1) / m.sum(1)
# نطبّع الطول
        out.append(torch.nn.functional.normalize(v, dim=1))
# مصفوفة
    return torch.cat(out).numpy()
# متجهات المستندات
D = embed(docs)
# الشكل
print(D.shape)


In [ ]:
# بحث بالمعنى
def search_dense(q, k=3):
# cosine مع كل مستند
    s = D @ embed([q])[0]
# أعلى k
    return [docs[i] for i in s.argsort()[::-1][:k]]
# أسئلة
for q in ["who has a hamstring injury", "which player moved to another club", "who is out for a long time"]:
# النتايج
    print(q, "->", search_dense(q))


## ٤. تركيب الـ prompt والتوليد

ناخد المستندات المسترجعة ونحطها في prompt، وبعدين الموديل يكتب الإجابة.

In [ ]:
# نبني الـ prompt
def build_prompt(q, k=2):
# المستندات المسترجعة
    ctx = "\n".join(search_dense(q, k))
# الحقائق ثم السؤال
    return "Facts:\n" + ctx + "\nQuestion: " + q + "\nAnswer:"
# مثال
p = build_prompt("Which player has a hamstring injury?")
# نطبع الـ prompt
print(p)


In [ ]:
# واجهة التوليد
from transformers import pipeline
# GPT-2 الصغير
gen = pipeline("text-generation", model="gpt2")
# توليد حتمي
o = gen(p, max_new_tokens=20, do_sample=False, pad_token_id=50256)[0]["generated_text"]
# الإجابة بس
print(o[len(p):])
